In [1]:
import os
import shutil
import uuid
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. LIMPIEZA PREVIA Y DESCARGA
print("Iniciando Re-Arquitectura: Solucionando Fuga de Datos (Cross-Domain Split)...")
!rm -rf /content/dataset_train /content/dataset_val /content/pklot_raw
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d blanderbuss/parking-lot-dataset --unzip -p /content/pklot_raw -q

# 2. CREACIÓN DE DIRECTORIOS ESTRICTOS
DIR_TRAIN = '/content/dataset_train'
DIR_VAL = '/content/dataset_val'

for d in [DIR_TRAIN, DIR_VAL]:
    os.makedirs(f'{d}/Occupied', exist_ok=True)
    os.makedirs(f'{d}/Empty', exist_ok=True)

# 3. SEPARACIÓN INTELIGENTE (POR DOMINIO/CÁMARA)
print("\nSeparando imágenes por dominio (Entrenamiento: PUCPR/UFPR04 | Validación: UFPR05)...")
train_count, val_count = 0, 0

for root, dirs, files in os.walk('/content/pklot_raw'):
    if 'PKLotSegmented' in root:
        for file in files:
            if file.lower().endswith('.jpg'):
                ruta_original = os.path.join(root, file)
                nombre_unico = str(uuid.uuid4())[:6] + "_" + file

                estado = 'Occupied' if 'Occupied' in root else 'Empty'

                # Aislar UFPR05 estrictamente para Validación
                if 'UFPR05' in root:
                    shutil.move(ruta_original, f'{DIR_VAL}/{estado}/{nombre_unico}')
                    val_count += 1
                else:
                    # PUCPR y UFPR04 para Entrenamiento
                    shutil.move(ruta_original, f'{DIR_TRAIN}/{estado}/{nombre_unico}')
                    train_count += 1

print(f"✅ Separación lista: {train_count} imágenes para entrenar, {val_count} para validar.")
print("Liberando RAM (Borrando RAW)...")
shutil.rmtree('/content/pklot_raw')

# 4. NUEVOS GENERADORES DE DATOS
print("\nConfigurando Pipeline de Datos Libre de Sesgos...")
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)
val_datagen = ImageDataGenerator(rescale=1./255)

cnn_train_gen = train_datagen.flow_from_directory(DIR_TRAIN, target_size=(64, 64), batch_size=32, class_mode='binary')
cnn_val_gen = val_datagen.flow_from_directory(DIR_VAL, target_size=(64, 64), batch_size=32, class_mode='binary')
print("[🚀] Sistema purgado y listo para entrenar.")

Iniciando Re-Arquitectura: Solucionando Fuga de Datos (Cross-Domain Split)...
Dataset URL: https://www.kaggle.com/datasets/blanderbuss/parking-lot-dataset
License(s): unknown

Separando imágenes por dominio (Entrenamiento: PUCPR/UFPR04 | Validación: UFPR05)...
✅ Separación lista: 1060132 imágenes para entrenar, 331570 para validar.
Liberando RAM (Borrando RAW)...

Configurando Pipeline de Datos Libre de Sesgos...
Found 1060132 images belonging to 2 classes.
Found 331570 images belonging to 2 classes.
[🚀] Sistema purgado y listo para entrenar.


In [2]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("Construyendo la arquitectura de la CNN...")

model_cnn = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print("\n[⏳] Iniciando el entrenamiento de validación cruzada... (Modo MVP)")
start_time = time.time()

history_cnn = model_cnn.fit(
    cnn_train_gen,
    steps_per_epoch=2000,
    validation_data=cnn_val_gen,
    validation_steps=500,
    epochs=5,
    callbacks=[early_stop]
)

training_time_seconds = time.time() - start_time

final_acc = history_cnn.history['val_accuracy'][-1]
final_prec = history_cnn.history['val_precision'][-1]
final_rec = history_cnn.history['val_recall'][-1]
final_f1 = 2 * (final_prec * final_rec) / (final_prec + final_rec) if (final_prec + final_rec) > 0 else 0.0

print("\n" + "="*50)
print("🚀 RESULTADOS FINALES DEL HITO 6 (ARQUITECTURA CORREGIDA)")
print("="*50)
print(f"⏱️ Tiempo:                  {training_time_seconds/60:.2f} minutos")
print(f"📊 Accuracy (Exactitud):    {final_acc:.4f}")
print(f"🎯 Precision (Precisión):   {final_prec:.4f}")
print(f"🔍 Recall (Exhaustividad):  {final_rec:.4f}")
print(f"⚖️ F1-Score:                {final_f1:.4f}")
print("="*50)

Construyendo la arquitectura de la CNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



[⏳] Iniciando el entrenamiento de validación cruzada... (Modo MVP)
Epoch 1/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 227s 109ms/step - accuracy: 0.9846 - loss: 0.0499 - precision: 0.9830 - recall: 0.9830 - val_accuracy: 0.8130 - val_loss: 0.7067 - val_precision: 0.7578 - val_recall: 0.9993
Epoch 2/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 215s 107ms/step - accuracy: 0.9942 - loss: 0.0211 - precision: 0.9931 - recall: 0.9941 - val_accuracy: 0.9063 - val_loss: 0.3416 - val_precision: 0.8636 - val_recall: 0.9979
Epoch 3/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 214s 107ms/step - accuracy: 0.9957 - loss: 0.0183 - precision: 0.9949 - recall: 0.9956 - val_accuracy: 0.7809 - val_loss: 0.7213 - val_precision: 0.7292 - val_recall: 0.9931
Epoch 4/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 212s 106ms/step - accuracy: 0.9958 - loss: 0.0161 - precision: 0.9953 - recall: 0.9954 - val_accuracy: 0.9099 - val_loss: 0.2783 - val_precision: 0.8683 - val_recall: 0.9971
Epoch 5/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 207s 104ms/step - accuracy: 0.9

In [ ]:
# 3. EXPORTAR MODELO PARA LA WEB (cerebro del proyecto)
# Ejecutar DESPUÉS del entrenamiento. Descarga el zip y súbelo a public/models/parking/ en el repo.

import shutil
!pip install -q tensorflowjs

import tensorflowjs as tfjs

EXPORT_DIR = '/content/parking_tfjs'
shutil.rmtree(EXPORT_DIR, ignore_errors=True)

model_cnn.save('/content/parking_cnn.keras')
tfjs.converters.save_keras_model(model_cnn, EXPORT_DIR)

shutil.make_archive('/content/parking_tfjs', 'zip', EXPORT_DIR)

from google.colab import files
files.download('/content/parking_tfjs.zip')

print("✅ Modelo exportado.")
print("1. Descomprime parking_tfjs.zip")
print("2. Copia model.json y los .bin a public/models/parking/ del proyecto")
print("3. Haz git push y la web usará la CNN real")